# TAME Training Dynamics & MoE Gating

Minimal reproduction of two diagnostic plots: MoE gate contributions per epoch
(mean ± std over seeds), and per-molecule element-wise gate weight distributions
on the validation set (last seed) -- do the experts collapse to hard 0/1 routing,
or mix softly?

Data: `data/tame_dynamics_histories.json` + `data/tame_dynamics_gates.npz`
(produced by `scripts/tame_training_dynamics.py`).

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

results_dir = Path("data")
with open(results_dir / "tame_dynamics_histories.json") as f:
    all_histories = json.load(f)

gates_data = np.load(results_dir / "tame_dynamics_gates.npz")
graph_gates = gates_data["graph_gates"]
text_gates = gates_data["text_gates"]
desc_gates = gates_data["desc_gates"]

In [ ]:
def pad_histories(hist_list, key):
    """Pads unequal epoch lengths with NaN so mean/std can be computed."""
    max_len = max(len(h[key]) for h in hist_list)
    padded = []
    for h in hist_list:
        arr = np.array(h[key], dtype=np.float32)
        pad_width = max_len - len(arr)
        padded.append(np.pad(arr, (0, pad_width), constant_values=np.nan))
    matrix = np.stack(padded)
    return np.nanmean(matrix, axis=0), np.nanstd(matrix, axis=0)

In [ ]:
g_mean, g_std = pad_histories(all_histories, "graph_gate")
t_mean, t_std = pad_histories(all_histories, "text_gate")
d_mean, d_std = pad_histories(all_histories, "desc_gate")
epochs = np.arange(1, len(g_mean) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, g_mean, label="Graph Branch", color="#10b981", linewidth=2)
ax.fill_between(epochs, g_mean - g_std, g_mean + g_std, color="#10b981", alpha=0.2)
ax.plot(epochs, t_mean, label="Text Branch", color="#8b5cf6", linewidth=2)
ax.fill_between(epochs, t_mean - t_std, t_mean + t_std, color="#8b5cf6", alpha=0.2)
ax.plot(epochs, d_mean, label="Descriptor Branch", color="#f59e0b", linewidth=2)
ax.fill_between(epochs, d_mean - d_std, d_mean + d_std, color="#f59e0b", alpha=0.2)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Mean Gate Weight", fontsize=12)
ax.set_ylim(0, 1.0)
ax.axhline(0.333, color="black", linestyle="--", alpha=0.5, label="Uniform (0.33)")
ax.legend(loc="upper right", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
g_gates_flat = np.array(graph_gates).flatten()
t_gates_flat = np.array(text_gates).flatten()
d_gates_flat = np.array(desc_gates).flatten()

sns.histplot(g_gates_flat, bins=30, ax=axes[0], color="#10b981", kde=True, legend=False)
axes[0].set_title("Graph Gate Distribution")
axes[0].set_xlabel("Gate Weight")

sns.histplot(t_gates_flat, bins=30, ax=axes[1], color="#8b5cf6", kde=True, legend=False)
axes[1].set_title("Text Gate Distribution")
axes[1].set_xlabel("Gate Weight")

sns.histplot(d_gates_flat, bins=30, ax=axes[2], color="#f59e0b", kde=True, legend=False)
axes[2].set_title("Descriptor Gate Distribution")
axes[2].set_xlabel("Gate Weight")

plt.tight_layout()
plt.show()